In [1]:
!pip install -q trimesh numpy scipy
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q Pillow torch torchvision


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 45.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00


In [2]:
import numpy as np
import trimesh
from scipy.spatial import cKDTree

def sample_points_from_mesh(mesh_path, num_points=10000):
    """Amostra pontos uniformemente da superfície do mesh."""
    mesh = trimesh.load(mesh_path, force='mesh')
    points, _ = trimesh.sample.sample_surface(mesh, num_points)
    return points

def normalize_point_cloud(points):
    """Centraliza e normaliza para esfera unitária."""
    centroid = points.mean(axis=0)
    points = points - centroid
    max_dist = np.max(np.linalg.norm(points, axis=1))
    points = points / max_dist
    return points

def compute_fscore(gt_points, pred_points, threshold=0.01):
    """
    Calcula F-Score entre dois point clouds.

    threshold: distância máxima para considerar um ponto como "correto"
    """
    # Distância de cada ponto predito ao GT mais próximo
    tree_gt = cKDTree(gt_points)
    dist_pred_to_gt, _ = tree_gt.query(pred_points)

    # Distância de cada ponto GT ao predito mais próximo
    tree_pred = cKDTree(pred_points)
    dist_gt_to_pred, _ = tree_pred.query(gt_points)

    # Precision: % de pontos preditos próximos ao GT
    precision = (dist_pred_to_gt < threshold).mean() * 100

    # Recall: % de pontos GT próximos ao predito
    recall = (dist_gt_to_pred < threshold).mean() * 100

    # F-Score
    if precision + recall > 0:
        fscore = 2 * precision * recall / (precision + recall)
    else:
        fscore = 0.0

    return fscore, precision, recall

# ====== USO ======
gt_mesh_path = '/content/drive/MyDrive/Ufma/ECP/CG/meshes_gt/objeto_gt.glb'
pred_mesh_path = '/content/drive/MyDrive/Ufma/ECP/CG/meshes_ours/objeto_ours.ply'

# Amostrar pontos
gt_points = sample_points_from_mesh(gt_mesh_path, num_points=10000)
pred_points = sample_points_from_mesh(pred_mesh_path, num_points=10000)

# Normalizar (importante para o threshold fazer sentido)
gt_points = normalize_point_cloud(gt_points)
pred_points = normalize_point_cloud(pred_points)

# Calcular F-Score
fscore, precision, recall = compute_fscore(gt_points, pred_points, threshold=0.01)
print(f"F-Score: {fscore:.1f}")
print(f"Precision: {precision:.1f}")
print(f"Recall: {recall:.1f}")


ValueError: string is not a file: `/content/drive/MyDrive/Ufma/ECP/CG/meshes_gt/objeto_gt.glb`

In [5]:
import torch
import clip
import numpy as np
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_clip, preprocess = clip.load("ViT-B/32", device=device)

def compute_clip_similarity(gt_img_path, pred_img_path):
    """Cosseno entre embeddings CLIP de duas imagens."""
    gt_img = preprocess(Image.open(gt_img_path).convert('RGB')).unsqueeze(0).to(device)
    pred_img = preprocess(Image.open(pred_img_path).convert('RGB')).unsqueeze(0).to(device)

    with torch.no_grad():
        gt_feat = model_clip.encode_image(gt_img)
        pred_feat = model_clip.encode_image(pred_img)
        # Normalizar
        gt_feat = gt_feat / gt_feat.norm(dim=-1, keepdim=True)
        pred_feat = pred_feat / pred_feat.norm(dim=-1, keepdim=True)

    similarity = (gt_feat @ pred_feat.T).item()
    return similarity * 100  # em percentual, como no paper

# ====== USO com 24 views renderizadas ======
import os

def evaluate_clip_24views(gt_render_dir, pred_render_dir):
    """Calcula CLIP similarity média sobre 24 views."""
    similarities = []
    for i in range(24):
        gt_path = os.path.join(gt_render_dir, f"{i}.png")
        pred_path = os.path.join(pred_render_dir, f"{i}.png")

        if os.path.exists(gt_path) and os.path.exists(pred_path):
            sim = compute_clip_similarity(gt_path, pred_path)
            similarities.append(sim)

    return np.mean(similarities)
def clip_similarity_vs_input(input_img_path, pred_render_dir):
    """Compara cada view renderizada do gerado com a imagem de entrada."""
    sims = []
    for i in range(24):
        pred = os.path.join(pred_render_dir, f"{i}.png")
        if os.path.exists(pred):
            sims.append(compute_clip_similarity(input_img_path, pred))
    return np.mean(sims)


# Exemplo (após renderizar com Blender):
# clip_score = evaluate_clip_24views("output/objeto_gt/render_512", "output/objeto_ours/render_512")
# print(f"CLIP Similarity: {clip_score:.1f}")


In [6]:
import os
import numpy as np

# ====== CONFIGURAÇÃO ======
base_dir = '/content/drive/MyDrive/Ufma/ECP/CG'

# Meshes de cada método (ajuste os caminhos)
gt_mesh = f'{base_dir}/meshes_gt/objeto_gt.glb'
meshes = {
    'One-2-3-45': f'{base_dir}/One-2-3-45-master/exp/busto-edited/mesh.ply',
    'Point-E': f'{base_dir}/resultados_point_e/objeto.ply',
    'Shap-E': f'{base_dir}/resultados_shap_e/objeto.ply',
}

# ====== F-SCORE ======
# print("="*50)
# print("F-SCORE (geometry)")
# print("="*50)

# gt_points = normalize_point_cloud(sample_points_from_mesh(gt_mesh))

# for metodo, mesh_path in meshes.items():
#     if os.path.exists(mesh_path):
#         pred_points = normalize_point_cloud(sample_points_from_mesh(mesh_path))
#         fscore, prec, rec = compute_fscore(gt_points, pred_points, threshold=0.01)
#         print(f"{metodo:15s} | F-Score: {fscore:.1f} | Precision: {prec:.1f} | Recall: {rec:.1f}")
#     else:
#         print(f"{metodo:15s} | Mesh não encontrado: {mesh_path}")

# ====== CLIP SIMILARITY (precisa das 24 views renderizadas) ======
render_dir = f'{base_dir}/One-2-3-45-master/exp/busto-edited'

print(f"\n{'='*50}")
print("CLIP SIMILARITY (appearance)")
print("="*50)

img_input = f'{base_dir}/One-2-3-45-master/inputs/edited/busto-edited.png'
render_dirs = {
    'One-2-3-45': f'{render_dir}/stage2_8',
    'Point-E': f'{render_dir}/objeto_point_e/render_512',
    'Shap-E': f'{render_dir}/objeto_shap_e/render_512',
}

for metodo, rdir in render_dirs.items():
    if os.path.exists(rdir):
        clip_score = clip_similarity_vs_input(img_input, rdir)
        print(f"{metodo:15s} | CLIP: {clip_score:.1f}")
    else:
        print(f"{metodo:15s} | Renders não encontrados (rode o Blender primeiro)")



CLIP SIMILARITY (appearance)
One-2-3-45      | Renders não encontrados (rode o Blender primeiro)
Point-E         | Renders não encontrados (rode o Blender primeiro)
Shap-E          | Renders não encontrados (rode o Blender primeiro)
